# 11. PPO (Proximal Policy Optimization)

**PPO（近位方策最適化）** は、OpenAI が 2017 年に発表した強化学習アルゴリズムで、現在も最も広く使われているオンポリシー手法の一つです。

## なぜ PPO が重要なのか？

前章（09_actor_critic.ipynb）で学んだ **A2C（Advantage Actor-Critic）** は、エピソード終了後に一括して方策を更新していました。
この方法にはいくつかの問題があります：

- **更新ステップが大きすぎる**：勾配降下が方策を急激に変化させ、学習が破綻することがある
- **サンプル効率が低い**：一回のエピソードデータを一度しか使わない（オンポリシーの宿命）

PPO はこれを解決するために **2 つの核心アイデア** を導入します：

| アイデア | 目的 |
|---|---|
| **Clipped Surrogate Objective** | 方策が古い方策から大きく離れないように制約する |
| **GAE（Generalized Advantage Estimation）** | Advantage の推定を λ でバイアス・分散のトレードオフを調整 |

## A2C との関係

PPO は A2C の直接的な発展形です：

```
REINFORCE → A2C → PPO
   (MC リターン)   (Advantage + Critic)   (Clipped 目標 + GAE + 複数エポック更新)
```

PPO は同じロールアウトデータを **複数エポック（K epochs）** 繰り返し使えるため、A2C よりもサンプル効率が高くなります。

## セットアップ

In [ ]:
import sys
import os

# rlpg パッケージを読み込めるようにパスを追加
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn

from src.environments.pendulum import InvertedPendulumEnv
from src.policies.ppo_policy import PPOPolicy, PPOActorCriticNet, RolloutBuffer

print('インポート完了')
print(f'PyTorch バージョン: {torch.__version__}')
print(f'デバイス: CPU（GPU不要）')

## 1. PPO の核心アイデア

### 1-1. Clipped Surrogate Objective

A2C の方策勾配目的関数：
$$J_{A2C}(\theta) = \mathbb{E}_t \left[ \log \pi_\theta(a_t | s_t) \cdot A_t \right]$$

PPO は「古い方策 $\pi_{\theta_{old}}$ に対する新しい方策の確率比」を使います：
$$r_t(\theta) = \frac{\pi_\theta(a_t | s_t)}{\pi_{\theta_{old}}(a_t | s_t)}$$

**PPO-Clip 目的関数：**
$$L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta) A_t,\; \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon) A_t \right) \right]$$

ここで $\varepsilon = 0.2$ が典型値です。

**直感的な理解：**
- $r_t = 1$：新旧方策が同じ → 通常の方策勾配更新と同じ
- $r_t > 1 + \varepsilon$：新方策が大きく変化 → クリップして更新を制限
- $r_t < 1 - \varepsilon$：新方策が逆方向に大きく変化 → クリップして更新を制限

### 1-2. GAE（Generalized Advantage Estimation）

TD 残差（1ステップ先読み）：
$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

GAE による Advantage 推定：
$$A_t^{GAE(\lambda)} = \sum_{l=0}^{T} (\gamma \lambda)^l \cdot \delta_{t+l}$$

$\lambda$ パラメータによるバイアス・分散のトレードオフ：

| $\lambda$ | 推定方式 | バイアス | 分散 |
|---|---|---|---|
| $\lambda = 0$ | TD(0)（1ステップ先読み） | 高（V の近似誤差に依存） | 低 |
| $\lambda = 1$ | モンテカルロ（エピソード全体） | 低 | 高 |
| $\lambda = 0.95$ | 実践的なバランス値（推奨） | 中 | 中 |

### 1-3. A2C vs PPO の比較

| 観点 | A2C | PPO |
|---|---|---|
| 目的関数 | $\log \pi(a|s) \cdot A$ | Clipped Surrogate Objective |
| Advantage 推定 | $G_t - V(s_t)$（MC または TD） | GAE（$\gamma\lambda$ 重み付き TD） |
| ネットワーク | Actor + Critic（分離） | Actor-Critic（共有ネット） |
| ロールアウト再利用 | 1 エポックのみ | K エポック繰り返す |
| 更新の安定性 | 低（ステップサイズ依存） | 高（クリッピングで制約） |
| 実装の複雑さ | 低 | 中 |

### クリッピングの視覚的説明

In [ ]:
# PPO のクリッピング機構を可視化する
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('PPO Clipped Surrogate Objective の仕組み', fontsize=13, fontweight='bold')

r = np.linspace(0.5, 1.5, 300)  # 確率比 r_t
eps = 0.2

for ax_idx, A in enumerate([1.0, -1.0]):
    ax = axes[ax_idx]

    surr1 = r * A                                             # 通常の目標
    surr2 = np.clip(r, 1 - eps, 1 + eps) * A                # クリップ後の目標
    ppo_obj = np.minimum(surr1, surr2)                       # PPO 目標（min を取る）

    ax.plot(r, surr1,   label=r'$r_t \cdot A_t$ (クリップなし)', lw=2, ls='--', color='steelblue')
    ax.plot(r, surr2,   label=r'$\mathrm{clip}(r_t) \cdot A_t$', lw=2, ls=':', color='darkorange')
    ax.plot(r, ppo_obj, label=r'$L^{CLIP}$（PPO 目標）', lw=3, color='crimson')

    ax.axvline(1 - eps, color='gray', ls='--', alpha=0.5, label=f'r = 1±{eps}')
    ax.axvline(1 + eps, color='gray', ls='--', alpha=0.5)
    ax.axvline(1.0, color='black', ls='-', alpha=0.3, lw=1)

    sign = '+' if A > 0 else '-'
    ax.set_title(f'A = {sign}1（Advantage が{"正" if A > 0 else "負"}の場合）')
    ax.set_xlabel('確率比 $r_t = \\pi_\\theta / \\pi_{\\theta_{old}}$')
    ax.set_ylabel('目的関数の値')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('解釈:')
print('  A > 0（良い行動）: r が 1+ε を超えたら目標をクリップ → 方策を急激に変えない')
print('  A < 0（悪い行動）: r が 1-ε を下回ったら目標をクリップ → 急激に方策を変えない')

## 2. PPOPolicy の実装確認

`src/policies/ppo_policy.py` に実装された `PPOPolicy` クラスを確認します。

In [ ]:
# PPOPolicy のインスタンス化
policy = PPOPolicy(
    hidden_sizes=[64, 64],    # 共有ネットワークの隠れ層
    action_low=-10.0,         # 倒立振子の最小力 [N]
    action_high=10.0,         # 倒立振子の最大力 [N]
    gamma=0.99,               # 割引率
    gae_lambda=0.95,          # GAE の λ（バイアス・分散トレードオフ）
    clip_eps=0.2,             # PPO クリッピング ε
    lr=3e-4,                  # Adam 学習率
    n_epochs=10,              # 1 ロールアウトあたりの更新エポック数
    mini_batch_size=64,       # ミニバッチサイズ
    value_coef=0.5,           # 価値損失の重み
    entropy_coef=0.01,        # エントロピーボーナスの重み（探索促進）
    max_grad_norm=0.5,        # 勾配クリッピング
)

print('=== PPOPolicy ===' )
print(policy)
print()
print('=== 共有 Actor-Critic ネットワーク ===')
print(policy.net)
print()
print(f'総パラメータ数: {policy.get_num_params():,}')

In [ ]:
# ネットワークの各層のパラメータ数を確認
print('=== 各モジュールのパラメータ数 ===')
for name, module in policy.net.named_modules():
    params = sum(p.numel() for p in module.parameters())
    if params > 0 and not isinstance(module, nn.Sequential):
        print(f'  {name:30s}: {params:6,} パラメータ')

print()
print('=== A2C との構造比較 ===')
print('A2C : Actor ネット + Critic ネット（分離）')
print('PPO : trunk（共有）→ actor_head + critic_head（分岐）')
print('      共有トランクにより特徴表現を効率的に学習')

## 3. 学習ループ

PPO の学習ループは A2C とは異なります。**ロールアウト収集** と **ポリシー更新** を分離して行います。

```
for iteration in range(N_ITER):
    # 1. ロールアウト収集（T ステップ）
    for t in range(ROLLOUT_STEPS):
        action, log_prob, value = policy.get_action_train(state)
        next_state, reward, done, _ = env.step(action)
        policy.push(state, action, log_prob, reward, done, value)
        ...

    # 2. GAE でアドバンテージを計算し、K エポック更新
    stats = policy.update(last_value=last_v)
```

A2C との重要な違い：
- ロールアウトを **T=512 ステップ** 分まとめて収集してから更新（複数エピソードを跨ぐ）
- 同じデータを **K=10 エポック** 繰り返して更新（サンプル効率の改善）
- 更新後は `buffer.clear()` でバッファをリセット（オンポリシー）

In [ ]:
# ---- 学習ハイパーパラメータ ----
N_ITER       = 200    # 学習イテレーション数
ROLLOUT_STEPS = 512   # 1 イテレーションで収集するステップ数
MAX_STEPS     = 500   # 1 エピソードの最大ステップ数
LOG_EVERY     = 20    # ログ出力頻度

# ---- 環境とポリシーの初期化 ----
env = InvertedPendulumEnv(max_steps=MAX_STEPS)

policy = PPOPolicy(
    hidden_sizes=[64, 64],
    action_low=-10.0,
    action_high=10.0,
    gamma=0.99,
    gae_lambda=0.95,
    clip_eps=0.2,
    lr=3e-4,
    n_epochs=10,
    mini_batch_size=64,
    value_coef=0.5,
    entropy_coef=0.01,
    max_grad_norm=0.5,
)

# ---- ログ用リスト ----
episode_rewards   = []   # 完了したエピソードの累積報酬
episode_lengths   = []   # 完了したエピソードの長さ
policy_loss_hist  = []   # イテレーションごとの policy_loss
value_loss_hist   = []   # イテレーションごとの value_loss
entropy_hist      = []   # イテレーションごとの entropy

# ---- 初期リセット ----
state = env.reset()
ep_reward = 0.0
ep_length = 0

print(f'学習開始: {N_ITER} イテレーション × {ROLLOUT_STEPS} ステップ/イテレーション')
print(f'合計ステップ数: {N_ITER * ROLLOUT_STEPS:,}')
print()

In [ ]:
# ---- PPO 学習ループ ----
for iteration in range(1, N_ITER + 1):

    # ---- Phase 1: ロールアウト収集 ----
    iter_episodes = []   # このイテレーション内で完了したエピソードの報酬

    for t in range(ROLLOUT_STEPS):
        # 確率的行動選択（訓練時）
        action, log_prob, value = policy.get_action_train(state)

        # 環境ステップ
        next_state, reward, done, _ = env.step(action)

        # ロールアウトバッファに格納
        policy.push(state, action, log_prob, reward, done, value)

        ep_reward += reward
        ep_length += 1

        if done:
            # エピソード完了 → リセット
            iter_episodes.append(ep_reward)
            episode_rewards.append(ep_reward)
            episode_lengths.append(ep_length)
            ep_reward = 0.0
            ep_length = 0
            state = env.reset()
        else:
            state = next_state

    # ---- Phase 2: ポリシー更新 ----
    # ロールアウト最後の状態の価値推定（エピソード途中なら V(s_T)、終了なら 0）
    if not done:
        # ロールアウトが途中で切れた場合：ブートストラップ
        _, _, last_value = policy.get_action_train(state)
    else:
        last_value = 0.0

    # GAE 計算 + K エポック更新
    stats = policy.update(last_value=last_value)

    # 損失を記録
    policy_loss_hist.append(stats['policy_loss'])
    value_loss_hist.append(stats['value_loss'])
    entropy_hist.append(stats['entropy'])

    # ---- ログ出力 ----
    if iteration % LOG_EVERY == 0:
        recent_rewards = episode_rewards[-20:] if len(episode_rewards) >= 20 else episode_rewards
        avg_r = np.mean(recent_rewards) if recent_rewards else 0.0
        print(
            f'Iter {iteration:4d}/{N_ITER} | '
            f'avg_reward={avg_r:7.1f} | '
            f'policy_loss={stats["policy_loss"]:7.4f} | '
            f'value_loss={stats["value_loss"]:7.4f} | '
            f'entropy={stats["entropy"]:6.4f}'
        )

print('\n学習完了')

## 4. 学習曲線のプロット

In [ ]:
def smooth(data, window=20):
    """移動平均でデータを平滑化する"""
    if len(data) < window:
        return np.array(data)
    kernel = np.ones(window) / window
    return np.convolve(data, kernel, mode='valid')


fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('PPO 学習曲線 — 倒立振子', fontsize=14, fontweight='bold')

# ---- 1: エピソード報酬 ----
ax = axes[0, 0]
ep_x = np.arange(1, len(episode_rewards) + 1)
sw = min(20, max(1, len(episode_rewards) // 5))
ax.plot(ep_x, episode_rewards, alpha=0.3, color='royalblue', label='報酬（生）')
if len(episode_rewards) >= sw:
    sm = smooth(episode_rewards, sw)
    ax.plot(ep_x[sw - 1:], sm, color='royalblue', lw=2, label=f'移動平均({sw})')
ax.axhline(MAX_STEPS, ls='--', color='gray', alpha=0.6, label=f'最大 {MAX_STEPS}')
ax.set_xlabel('エピソード番号')
ax.set_ylabel('累積報酬')
ax.set_title('エピソード報酬')
ax.legend()
ax.grid(alpha=0.3)

# ---- 2: エピソード長 ----
ax = axes[0, 1]
ax.plot(ep_x, episode_lengths, alpha=0.3, color='darkorange')
if len(episode_lengths) >= sw:
    sm_len = smooth(episode_lengths, sw)
    ax.plot(ep_x[sw - 1:], sm_len, color='darkorange', lw=2, label=f'移動平均({sw})')
ax.axhline(MAX_STEPS, ls='--', color='gray', alpha=0.6, label=f'最大 {MAX_STEPS} ステップ')
ax.set_xlabel('エピソード番号')
ax.set_ylabel('ステップ数')
ax.set_title('エピソード長（バランス継続時間）')
ax.legend()
ax.grid(alpha=0.3)

# ---- 3: Policy Loss ----
ax = axes[1, 0]
iter_x = np.arange(1, len(policy_loss_hist) + 1)
ax.plot(iter_x, policy_loss_hist, alpha=0.4, color='crimson', label='policy_loss（生）')
sw2 = min(20, max(1, len(policy_loss_hist) // 5))
if len(policy_loss_hist) >= sw2:
    sm_pl = smooth(policy_loss_hist, sw2)
    ax.plot(iter_x[sw2 - 1:], sm_pl, color='crimson', lw=2, label=f'移動平均({sw2})')
ax.set_xlabel('イテレーション')
ax.set_ylabel('Policy Loss')
ax.set_title('PPO Policy Loss（Clipped Surrogate）')
ax.legend()
ax.grid(alpha=0.3)

# ---- 4: Value Loss ----
ax = axes[1, 1]
ax.plot(iter_x, value_loss_hist, alpha=0.4, color='teal', label='value_loss（生）')
if len(value_loss_hist) >= sw2:
    sm_vl = smooth(value_loss_hist, sw2)
    ax.plot(iter_x[sw2 - 1:], sm_vl, color='teal', lw=2, label=f'移動平均({sw2})')
ax.set_xlabel('イテレーション')
ax.set_ylabel('Value Loss')
ax.set_title('Value Loss（MSE）')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 最終的な統計情報
print('=== 学習統計 ===')
print(f'完了エピソード数         : {len(episode_rewards)}')
print(f'最終20エピソードの平均報酬: {np.mean(episode_rewards[-20:]):.1f}')
print(f'最終20エピソードの平均長  : {np.mean(episode_lengths[-20:]):.1f} ステップ')
print(f'完走率（{MAX_STEPS}ステップ）  : {np.mean([l >= MAX_STEPS for l in episode_lengths[-20:]])*100:.0f}%')

## 5. 学習済みポリシーの評価

探索なしの確定的行動（`get_action()`）で評価します。  
訓練時の確率的行動選択（`get_action_train()`）とは異なり、アクター平均のみを使います。

In [ ]:
def evaluate_policy(policy: PPOPolicy, n_episodes: int = 10, max_steps: int = 500) -> dict:
    """学習済み PPO ポリシーを確定的行動で評価する"""
    env_eval = InvertedPendulumEnv(max_steps=max_steps)
    rewards = []
    lengths = []

    for ep in range(n_episodes):
        state = env_eval.reset()
        total_r, steps, done = 0.0, 0, False

        while not done:
            # 確定的行動（探索なし）：アクター平均を使用
            action = policy.get_action(state)
            state, r, done, _ = env_eval.step(action)
            total_r += r
            steps += 1

        rewards.append(total_r)
        lengths.append(steps)

    return {
        'rewards'     : rewards,
        'lengths'     : lengths,
        'mean_reward' : np.mean(rewards),
        'std_reward'  : np.std(rewards),
        'mean_length' : np.mean(lengths),
        'solved_rate' : np.mean([l >= max_steps for l in lengths]),
    }


eval_result = evaluate_policy(policy, n_episodes=10, max_steps=MAX_STEPS)

print('=== 評価結果（確定的行動、10 エピソード） ===')
print(f'  平均報酬    : {eval_result["mean_reward"]:8.1f} ± {eval_result["std_reward"]:.1f}')
print(f'  平均ステップ: {eval_result["mean_length"]:8.1f} / {MAX_STEPS}')
print(f'  完走率      : {eval_result["solved_rate"]*100:.0f}%  ({MAX_STEPS} ステップ完走)')
print()
print('各エピソードの報酬:')
for i, (r, l) in enumerate(zip(eval_result['rewards'], eval_result['lengths']), 1):
    bar = '#' * int(l / MAX_STEPS * 20)
    solved = '（完走）' if l >= MAX_STEPS else ''
    print(f'  エピソード {i:2d}: {r:6.1f} 報酬 / {l:3d} ステップ  {bar:<20} {solved}')

## 6. A2C との比較

### アルゴリズムの違い

**A2C（09章）** と **PPO（本章）** は同じ Actor-Critic の枠組みに基づきますが、以下の点で重要な違いがあります：

#### 目的関数

```
A2C:
  policy_loss = -log π(a|s) * A_t
  （方策勾配をそのまま使用）

PPO:
  r_t = π_θ(a|s) / π_θ_old(a|s)
  policy_loss = -min(r_t * A_t, clip(r_t, 1-ε, 1+ε) * A_t)
  （クリッピングで更新幅を制約）
```

#### Advantage 推定

```
A2C:  A_t = G_t - V(s_t)         モンテカルロ or TD(1)
PPO:  A_t = GAE(λ)               λ 重み付き TD 残差の和（分散低減）
```

#### 更新回数

```
A2C:  1 エピソード → 1 回更新（データを捨てる）
PPO:  T ステップロールアウト → K エポック × M ミニバッチ更新
      → 同じデータを K=10 回再利用してサンプル効率を改善
```

### 実用上の特性比較

| 特性 | A2C | PPO |
|---|---|---|
| 実装の簡単さ | 容易 | やや複雑 |
| サンプル効率 | 低（1 回のみ使用） | 高（K エポック再利用） |
| 学習の安定性 | 中（学習率依存） | 高（クリッピングで保護） |
| ハイパーパラメータ数 | 少ない | やや多い（ε, λ, K など） |
| 最終的な性能 | 良い | 同等以上（多くのタスクで） |
| 並列環境への拡張 | A3C として自然 | PPO + 並列環境が標準的 |

### なぜ PPO が現在の標準なのか

PPO は **シンプルさ** と **安定性** と **性能** のバランスが優れています：
- TRPO（Trust Region Policy Optimization）の安定性を持ちながら実装が簡単
- OpenAI Five（Dota 2）、AlphaStar（StarCraft II）など大規模タスクでも実績あり
- ロボット制御、ゲームプレイ、自然言語処理（RLHF）など幅広い応用

## 7. まとめ

### 本章で学んだこと

| 概念 | 説明 |
|---|---|
| **Clipped Surrogate Objective** | 確率比 $r_t$ をクリッピングし、方策が急激に変わることを防ぐ |
| **GAE** | $\lambda$ で TD(0) とモンテカルロのトレードオフを調整し、Advantage の分散を低減 |
| **共有 Actor-Critic ネット** | trunk を共有し、actor_head と critic_head に分岐することでパラメータを削減 |
| **ロールアウト収集と複数エポック更新** | T ステップのロールアウトを K エポック再利用してサンプル効率を改善 |
| **ブートストラップ** | ロールアウト途中でエピソードが終わらなくても V(s_T) で last_value を補完 |

### PPO の学習フロー（復習）

```
繰り返し（イテレーション）:
  ┌── ロールアウト収集フェーズ ──────────────────────────────┐
  │  T ステップ間、get_action_train() で行動を選択             │
  │  push(state, action, log_prob, reward, done, value)     │
  └─────────────────────────────────────────────────────────┘
  ┌── 更新フェーズ ──────────────────────────────────────────┐
  │  GAE で Advantage と Returns を計算                       │
  │  Advantage を正規化（ゼロ平均・単位分散）                 │
  │  K エポック × M ミニバッチ:                               │
  │    ratio = exp(new_log_prob - old_log_prob)              │
  │    policy_loss = -min(ratio * A, clip(ratio) * A).mean()│
  │    value_loss = MSE(V_pred, returns)                     │
  │    loss = policy_loss + 0.5 * value_loss - 0.01 * H     │
  │    optimizer.step()                                      │
  │  buffer.clear()                                          │
  └─────────────────────────────────────────────────────────┘
```

### 次のステップ（12章: SAC）

PPO は **オンポリシー** アルゴリズムです。収集したデータを更新後は捨てるため、データ効率に限界があります。

次章では **SAC（Soft Actor-Critic）** を学びます：

| 観点 | PPO | SAC |
|---|---|---|
| ポリシー種別 | オンポリシー | オフポリシー |
| データ再利用 | K エポックのみ | リプレイバッファで大量再利用 |
| エントロピー最大化 | 補助項（小さい重み） | 目的関数の核心（自動温度調整） |
| サンプル効率 | 中 | 高 |
| 連続行動空間 | 対応 | 設計上最適 |

SAC はエントロピー正則化を明示的に最大化することで、**探索と利用のバランス** をより洗練された形で実現します。